# 019 — Lógica proposicional e inferencia

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("logic", seed=19)
assert result["kind"] == "logic"
assert result["evidence"]
show(result)


## Solución 1 — Tablas

| P | Q | P→Q | KB₁={P→Q, P} | Q |
|---|---|-----|--------------|---|
| F | F | V | F | — |
| F | V | V | F | — |
| V | F | F | F | — |
| V | V | V | **V** | **V** |

La única fila donde KB₁ es verdadera tiene Q verdadera → `⊨ Q` ✔.
Para KB₂={P→Q, Q}: es verdadera en la fila (P=F, Q=V), donde P es **falsa** —
esa fila es el contraejemplo: `⊭ P`.


In [ ]:
from itertools import product
contraejemplos = [(P, Q) for P, Q in product([False, True], repeat=2)
                  if ((not P) or Q) and Q and not P]
assert contraejemplos == [(False, True)]
print("contraejemplo de afirmar el consecuente:", contraejemplos[0])


## Solución 2 — CNF

a) `(P → Q) → R`
⇒ `¬(¬P ∨ Q) ∨ R` (eliminar →)
⇒ `(P ∧ ¬Q) ∨ R` (De Morgan + doble negación)
⇒ **`(P ∨ R) ∧ (¬Q ∨ R)`** (distribuir ∨ sobre ∧).

b) `¬(P ∨ (Q ∧ R))`
⇒ `¬P ∧ ¬(Q ∧ R)` (De Morgan)
⇒ `¬P ∧ (¬Q ∨ ¬R)` (De Morgan) — ya es CNF: **dos cláusulas**, `{¬P}` y `{¬Q, ¬R}`.


## Solución 3 — Refutación

Cláusulas: `1: P∨Q`, `2: ¬P∨R`, `3: ¬Q∨R`, `4: ¬R` (negación de la meta).

```text
5: ¬P     (resolver 2 y 4 sobre R)
6: ¬Q     (resolver 3 y 4 sobre R)
7: Q      (resolver 1 y 5 sobre P)
8: □      (resolver 7 y 6 sobre Q)   → contradicción
```

Derivada la cláusula vacía, `KB ∧ ¬R` es insatisfacible, luego `KB ⊨ R` ✔.


## Solución 4 — La cadena de modus ponens

a) `tiene_datos ∧ tiene_objetivo → puede_experimentar`;
`puede_experimentar → requiere_baseline`;
`requiere_baseline → requiere_evaluacion`.

b) Disparo 1: los hechos iniciales `tiene_datos` y `tiene_objetivo` habilitan
la primera regla (modus ponens con premisa conjunta). Disparo 2: el hecho
recién derivado `puede_experimentar` habilita la segunda. Disparo 3:
`requiere_baseline` habilita la tercera. Cada conclusión del JSON conserva su
regla (`if`/`then`): esa lista ES la prueba formal, legible paso a paso.


In [ ]:
result = run_lab("logic", seed=19)
fired = result["result"]["rules_fired"]
assert [f["then"] for f in fired] == [
    "puede_experimentar", "requiere_baseline", "requiere_evaluacion"]
print("cadena de modus ponens verificada ✔")


## Reflexión

1. El laboratorio deriva `requiere_evaluacion` por encadenamiento hacia adelante. Escribe su base de reglas como cláusulas de Horn y verifica que cada disparo es una aplicación de modus ponens.
2. La tabla de verdad decide la implicación en 2^n filas y la resolución evita enumerarlas, pero SAT es NP-completo. ¿Qué gana entonces la resolución en la práctica?
3. ¿Por qué el encadenamiento hacia adelante es completo para cláusulas de Horn pero no para lógica proposicional arbitraria? Da un ejemplo de fórmula que no pueda representarse como Horn.
